# Watch a policy optimize the wrong reward

Day 24 · Chapter 14 · CPU worked notebook

Read Chapter14 sections14.1–14.2. This finite policy is trained with exact expectation, so sampling noise is removed.


### Prediction

Predict: which response wins if rewards are1 for35,2 for35 0,0 for0? What happens to strict accuracy?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


### Prediction

Controlled change: use rewards1,1,0 and predict whether both accepted actions retain mass. Repairing the reward from equal initial logits differs from repairing a previously hacked checkpoint. Evidence boundary: an exact three-action policy demonstrates misspecification; no claim of language-model repair.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
from dongxi_llms.optimization_diagnostics_lab import hacking_experiment
broken = hacking_experiment()
fixed = hacking_experiment(repair=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for result, name in ((broken, "Broken proxy"), (fixed, "Strict reward restart")):
    steps = [r["update"] for r in result["history"]]
    axes[0].plot(steps, [r["proxy_reward"] for r in result["history"]], label=name)
    axes[1].plot(steps, [r["strict_accuracy"] for r in result["history"]], label=name)
axes[0].set(xlabel="SGD update", ylabel="Expected programmed reward")
axes[1].set(xlabel="SGD update", ylabel="Probability of strict correct answer")
for ax in axes: ax.legend()
plt.show()
print("Actions:", broken["responses"])

### Reference explanation

Reading guide: the proxy curves use different reward definitions, while strict accuracy uses one common criterion. The broken policy's rising proxy can accompany falling correctness.

**Exercise modification reference.** 

```python
z = torch.nn.Parameter(torch.zeros(3, dtype=torch.float64))
reward = torch.tensor([1., 1., 0.], dtype=torch.float64)
optimizer = torch.optim.SGD([z], lr=.4)
for _ in range(60):
    optimizer.zero_grad(set_to_none=True)
    loss = -(z.softmax(-1)*reward).sum()
    loss.backward()
    optimizer.step()
p = z.detach().softmax(-1)
print('Probabilities:', p.tolist(), 'strict one-integer accuracy:', float(p[0]))
torch.testing.assert_close(p[0], p[1])
```

Equal initialization plus equal rewards preserve equal probability for35 and35 0. Both receive proxy credit; only the first satisfies the unchanged strict task. This fresh symmetric variant does not repair an already hacked checkpoint. This concrete CPU variant has not been executed by the unchanged reference cell; it supplies an exercise reference rather than a new measured result.


Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Compare this fixed preview with your prediction and the current plot.

![Saved reference 1: reward hacking](../figures/chapter-14/day-24-01_reward_hacking-01.png)

### Prediction

Exercise: derive dJ/dz_i = p_i(R_i-J) and explain why the highest proxy action is favored.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
# Reference solution verifies the derivative against autograd.
z = torch.zeros(3, dtype=torch.float64, requires_grad=True)
reward = torch.tensor([1., 2., 0.], dtype=torch.float64)
p = z.softmax(-1)
J = (p*reward).sum()
J.backward()
expected = p.detach()*(reward-J.detach())
torch.testing.assert_close(z.grad, expected)
print("Ascent direction:", z.grad.tolist())
print("Strict repair final:", fixed["history"][-1])

### Reference explanation

Autograd agrees with `p_i (R_i − J)` for the explicit reward vector `[1,2,0]`. Gradient ascent therefore favors the overrewarded malformed action. The `[1,1,0]` equal-correctness variant is a proposed modification, not an observed repair result from the unchanged cell.
